# Image

In [2]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [3]:
model_path = "./pose_landmarker.task"

In [4]:
import mediapipe as mp

BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    running_mode=VisionRunningMode.IMAGE)

mp_image = mp.Image.create_from_file('./image.jpg')

with PoseLandmarker.create_from_options(options) as landmarker:
  # The landmarker is initialized. Use it here.
  # ...
  pose_landmarker_result = landmarker.detect(mp_image)
    

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1769700365.886620   16772 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1769700366.040360   16772 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1769700366.189437   16772 landmark_projection_calculator.cc:78] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


In [ ]:
import cv2
import numpy as np
import os
import sys


def has_display():
    """Return True if a GUI window can actually be shown (X11/Wayland/Windows/macOS)."""
    if os.name == "nt" or sys.platform == "darwin":
        return True
    return bool(os.environ.get("DISPLAY") or os.environ.get("WAYLAND_DISPLAY"))


def visualize_qb_manual(image_path, detection_result):
    image = cv2.imread(image_path)
    h, w, _ = image.shape

    if not detection_result.pose_landmarks:
        return image

    # MediaPipe Pose Connections (Symmetry for QB: Shoulder to Elbow, Elbow to Wrist)
    # These are the indices for the right arm: 12 (Shoulder), 14 (Elbow), 16 (Wrist)
    CONNECTIONS = [(12, 14), (14, 16), (11, 13), (13, 15), (12, 11), (24, 23)]

    for pose_landmarks in detection_result.pose_landmarks:
        # 1. Draw joints
        for lm in pose_landmarks:
            cx, cy = int(lm.x * w), int(lm.y * h)
            cv2.circle(image, (cx, cy), 5, (0, 255, 0), -1)

        # 2. Draw skeleton lines
        for start_idx, end_idx in CONNECTIONS:
            start_lm = pose_landmarks[start_idx]
            end_lm = pose_landmarks[end_idx]

            start_point = (int(start_lm.x * w), int(start_lm.y * h))
            end_point = (int(end_lm.x * w), int(end_lm.y * h))

            cv2.line(image, start_point, end_point, (255, 0, 0), 2)

    return image

# Run it
result_img = visualize_qb_manual('image.jpg', pose_landmarker_result)

cv2.imwrite('output_image.jpg', result_img)
print("Saved annotated image to output_image.jpg")

if has_display():
    cv2.imshow('Manual QB Trace', result_img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
else:
    print("No display detected (headless environment) - skipping cv2.imshow.")


# Video

In [ ]:
import cv2
import numpy as np

def visualize_qb_analysis(frame, detection_result):
    """
    frame: The BGR numpy array from cv2.VideoCapture
    detection_result: The result object from detector.detect_for_video
    """
    # Create a copy so we don't overwrite the original frame
    canvas = frame.copy()
    h, w, _ = canvas.shape

    # If no landmarks, return the original frame
    if not detection_result or not detection_result.pose_landmarks:
        return canvas

    # Indices for the throwing arm (Right Side)
    # 12: Shoulder, 14: Elbow, 16: Wrist
    SHOULDER = 12
    ELBOW = 14
    WRIST = 16

    for pose_landmarks in detection_result.pose_landmarks:
        # 1. Convert normalized coordinates to pixel coordinates
        def get_pix(idx):
            lm = pose_landmarks[idx]
            return (int(lm.x * w), int(lm.y * h))

        p_shoulder = get_pix(SHOULDER)
        p_elbow = get_pix(ELBOW)
        p_wrist = get_pix(WRIST)

        # 2. Draw the "Throwing Triangle"
        # Draw lines between joints
        cv2.line(canvas, p_shoulder, p_elbow, (255, 255, 255), 2)
        cv2.line(canvas, p_elbow, p_wrist, (255, 255, 255), 2)

        # Draw the joint points
        for pt in [p_shoulder, p_elbow, p_wrist]:
            cv2.circle(canvas, pt, 6, (0, 255, 0), -1)

        # 3. Calculate Elbow Angle (The Biomechanics)
        # Vector BA (Elbow to Shoulder) and Vector BC (Elbow to Wrist)
        ba = np.array(p_shoulder) - np.array(p_elbow)
        bc = np.array(p_wrist) - np.array(p_elbow)

        cosine_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc))
        angle = np.degrees(np.arccos(np.clip(cosine_angle, -1.0, 1.0)))

        # 4. Annotate the angle on the frame, next to the elbow
        label = f"{angle:.1f} deg"
        text_origin = (p_elbow[0] + 15, p_elbow[1])
        cv2.putText(canvas, label, text_origin, cv2.FONT_HERSHEY_SIMPLEX,
                    0.8, (0, 255, 255), 2, cv2.LINE_AA)

    return canvas


In [ ]:
import mediapipe as mp


BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

# Create a pose landmarker instance with the video mode:
options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    running_mode=VisionRunningMode.VIDEO)

with PoseLandmarker.create_from_options(options) as landmarker:

    video_path = 'qb-throw.mp4'
    output_path = 'qb-throw-annotated.mp4'
    cap = cv2.VideoCapture(video_path)

    # 2. Load the frame rate (FPS) and frame size
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps == 0: fps = 30  # Fallback if metadata is missing

    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))

    show_live = has_display()
    frame_count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # 3. Calculate timestamp in milliseconds
        # This is required by the MediaPipe Tasks 'video' mode
        timestamp_ms = int((frame_count / fps) * 1000)

        # 4. Convert BGR (OpenCV) to RGB (MediaPipe)
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # 5. Create MediaPipe Image object
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

        # 6. Detect Pose (Using video mode 'detect_for_video')
        pose_landmarker_result = landmarker.detect_for_video(mp_image, timestamp_ms)

        # 7. Draw the throwing triangle + elbow angle, then save the frame
        annotated_frame = visualize_qb_analysis(frame, pose_landmarker_result)
        writer.write(annotated_frame)

        if show_live:
            cv2.imshow('QB Biomechanics Feed', annotated_frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

        frame_count += 1

    cap.release()
    writer.release()
    if show_live:
        cv2.destroyAllWindows()

    print(f"Processed {frame_count} frames. Saved annotated video to {output_path}")
